# Plotting script

This script contains code that is used to generate various plots and figures to visualize training process and results. The generated plots are added to the final report and presentation slides.

## Plotting RLAgent loss


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import ast
import re
import numpy as np

plt.rcParams.update({'font.size': 18})

    
def normalize_vector(val):
    # Handle cases where it's already a number (like the 0s at the start)
    if pd.isna(val) or val == 0 or val == '0':
        return [None, None, None]
    
    
    # Handle the strings like "[0.2035 1.  0. ]"
    if isinstance(val, str):
        # Remove brackets and split by any whitespace
        clean = re.sub(r'[\[\]]', '', val).strip()
        parts = clean.split()
        
        if len(parts) == 3:
            res = []
            for x in parts:
                res.append(float(x)) if x != 0 else res.append(None)
            return res
            
    # Fallback for anything else
    return [None, None, None]


def plot_RLAgentLoss(dataset, approach, path, plotdir="plots", privacy_scalar=False, filename="RLAgentLoss.csv", metric=""):
    
    df = pd.read_csv(path)
    
    mask = df['step'].diff() < 0
    reset_indices = df.index[mask].tolist()

    # Extract the lengths of the different phases
    # pre_train_steps is the value just before the first reset
    pre_train_steps = df.loc[reset_indices[0] - 1, 'step']

    # steps_per_round is the value just before the second reset (if it exists)
    if len(reset_indices) > 1:
        steps_per_round = df.loc[reset_indices[1] - 1, 'step']
    else:
        # Fallback if the file only has one reset so far
        steps_per_round = pre_train_steps 

    # Create the 'resets' counter (0 for pre-train, 1 for round 1, 2 for round 2...)
    df['resets'] = mask.astype(int).cumsum()

    # Calculate total_step
    # Logic: If reset is 0, just use 'step'.
    # If reset > 0, add pre_train_steps + (remaining rounds * steps_per_round)
    df['total_step'] = np.where(
        df['resets'] == 0,
        df['step'],
        pre_train_steps + df['step'] + ((df['resets'] - 1) * steps_per_round)
    )

    # Optional: clean up the helper column
    df = df.drop(columns=['resets'])
    
    
    
    if not privacy_scalar:
        m_norm = df['mprivacy'].apply(normalize_vector)
        g_norm = df['gprivacy'].apply(normalize_vector)
        mprivacy = pd.DataFrame(m_norm.tolist(), columns=['dcr', 'nndr', 'gower'])
        gprivacy = pd.DataFrame(g_norm.tolist(), columns=['dcr', 'nndr', 'gower'])


    # Create the plot
    plt.figure(figsize=(10, 6))
    
    # Assuming columns are named 'epoch' (or 'step') and 'loss'
    # If files have different headers, change the names below
    plt.plot(df['total_step'], df['mloss'], label='M Loss', color='blue', linestyle=':', linewidth=2)
    plt.plot(df['total_step'], df['gloss'], label='G Loss', color='red', linewidth=2)
    
    if privacy_scalar:
        plt.plot(df['total_step'], df['mprivacy'], label='M privacy loss', color='#0B2CD6', linewidth=2)
        plt.plot(df['total_step'], df['gprivacy'], label='G privacy loss', color='pink', linewidth=2)
        
    else:
        plt.plot(df['total_step'], gprivacy['dcr'], label='G DCR Loss', color='green', linewidth=2)
        plt.plot(df['total_step'], gprivacy['nndr'], label='G NNDR Loss', color='orange', linewidth=2)
        plt.plot(df['total_step'], gprivacy['gower'], label='G Gower Loss', color='pink', linewidth=2)
        plt.plot(df['total_step'], mprivacy['dcr'], label='M DCR Loss', color='#27F542', linestyle=':', linewidth=2)
        plt.plot(df['total_step'], mprivacy['gower'], label='M Gower Loss', color='purple', linestyle=':', linewidth=2)
    
    # Add labels and title
    plt.title(f'{dataset}, {approach} {metric} Approach', fontsize=24)
    plt.xlabel('Step', fontsize=18)
    plt.ylabel('Loss Value', fontsize=18)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=18)

    # Show or save the plot
    plt.tight_layout()
    save_dir = os.path.join("../privacy_result", plotdir)
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
    plt.savefig(os.path.join(save_dir, f"{dataset}_{approach}{metric}_loss_plot.png")) # Saves the plot to a file


def plotting(dataset_names, approaches, plotdir="plots", privacy_scalar=False, filename="RLAgentLoss.csv", metric=""):
    for dataset in dataset_names:
        for approach in approaches:
            dir = f"../privacy_result/{dataset}/ddpm_cb_best_{approach}"
            path = os.path.join(dir, filename)
            if not os.path.exists(path):
                dir = f"../privacy_result/{dataset}/ddpm_mlp_best_{approach}"
                path = os.path.join(dir, "RLAgentLoss.csv")
            if not os.path.exists(path):
                print(f"File not found for dataset: {dataset}, approach: {approach}")
                continue
            print(f"Plotting {dataset}, approach: {approach}") 
            plot_RLAgentLoss(dataset=dataset, approach=approach, path=path, plotdir=plotdir, privacy_scalar=privacy_scalar, metric=metric)       

    
def plot_single_metric(dataset_names, approaches, plotdir="plots"):
    
    postfix_adaptive = ["DCR", "NNDR", "Gower"]
    postfix = ["dcr", "gower", "nndr"]
    
    for i in postfix_adaptive:
        filename = f"RLAgentLoss{i}.csv"
        plotting(dataset_names=dataset_names, approaches=approaches, plotdir=plotdir, privacy_scalar=True, filename=filename, metric=i)
    
    for i in postfix:
        filename = f"RLAgentLoss_{i}.csv"
        plotting(dataset_names=dataset_names, approaches=approaches, plotdir=plotdir, privacy_scalar=True, filename=filename, metric=i)


In [ ]:
dataset = ['abalone']
# dynamic adaptive trains with privacy loss only when the model is in a low privacy state
# dynamic_adaptive_consistent is always training with privacy loss
# dynamic_adaptive_lessnorm is to not force the model to have strictly decreasing privacy loss
approaches = ["dynamic_adaptive_consistent", "dynamic_adaptive"]
plots_dirname="plots/abalone_ablation"
plotting(dataset_names=dataset, approaches=approaches, plotdir=plots_dirname, privacy_scalar=False, filename="RLAgentLoss.csv", metric="")
plotting(dataset_names=dataset, approaches=["dynamic_sum"], plotdir=plots_dirname, privacy_scalar=True, filename="RLAgentLoss.csv", metric="")

In [ ]:
dataset_names = ['abalone', 'bike', 'buddy', 'california', 'churn2', 'gesture', 'insurance', 'loan', 'wilt', 'winequality']
approaches = ["multi_state"] # ["dynamic_adaptive_both", "dynamic_adaptive", "continuous_sum"] # 
plots_dirname = "plots/multi_state" # folder name for plots, located in privacy_result, 'plots' by default

# plotting(dataset_names, approaches, plotdir=plots_dirname, privacy_scalar=True)
# plot_RLAgentLoss('gesture','single_metric', "../privacy_result/gesture/ddpm_cb_best_single_metric/RLAgentLoss_nndr.csv", "plots/single_metric", True, "RLAgentLoss_nndr.csv", "nndr")
plot_single_metric(dataset_names=dataset_names, approaches=["adaptive_single_metric"], plotdir="plots/adaptive_single_metric")



## Plotting data quality

Comparing data quality of the synthetic data generated with various approaches


In [ ]:
import json
from typing import Union
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

plt.rcParams.update({'font.size': 18})

def load_eval_files(dataset, approaches) -> Union[dict, None]:
    possible_filenames = [
        "eval.txt",
        "eval_dcr.txt",
        "eval_nndr.txt",
        "eval_gower.txt",
        "SynTabRL_evaluation.txt",
        "SynTabRL_evaluation_dcr.txt",
        "SynTabRL_evaluation_gower.txt",
        "SynTabRL_evaluation_nndr.txt",
        "SynTabRL_eval.json",
    ]
    
    eval_dict = {}
    for approach in approaches:
        path = f"../privacy_result/{dataset}/ddpm_cb_best_{approach}/"
        existing_files = [
            os.path.join(path, f) for f in possible_filenames 
            if os.path.exists(os.path.join(path, f))
        ]
        path2 = f"../privacy_result/{dataset}/ddpm_mlp_best_{approach}/"
        existing_files2 = [
            os.path.join(path2, f) for f in possible_filenames 
            if os.path.exists(os.path.join(path2, f))
        ]
        existing_files.extend(existing_files2)
        if len(existing_files) == 0:
            print(f"No evaluation file found for dataset: {dataset}, approach: {approach}")
            return None
        for file in existing_files:
            if file.endswith(".json"):
                with open(file, 'r') as f:
                    eval_data = json.load(f)
                    eval_dict[approach] = eval_data
            else:
                with open(file, 'r') as f:
                    eval_data = {}
                    for line in f:
                        if ":" in line:
                            key, value = line.split(":", 1)
                            try:
                                eval_data[key.strip()] = float(value.strip())
                            except ValueError:
                                continue  # Skip lines where the value is not a valid float
                    eval_dict[approach] = eval_data    
                    
    return eval_dict    
    
    

def plot_two_approaches(app1, app2, eval_dict, dataset, comment="", no_categorical=False):

    # Define keys in order up to dcr_gower
    target_metrics = [
        'wasserstein_distance_numerical', 'js_similarity_categorical', 
        'correlation_pearson', 'correlation_spearman', 'dcr_euclidean', 
        'nndr_euclidean','gower_distance'
    ]
    if 'dcr_gower_distance' in eval_dict[app1]:
        target_metrics.remove('gower_distance')  # Remove the original gower_distance if dcr_gower_distance is present
        target_metrics.append('dcr_gower_distance')
    
    # Remove label of Gower to make space for legend    
    target_labels=['Num. Similarity', 'Cat. Similarity', 'Cor. Pearson', 
                   'Cor. Spearman', 'DCR', 'NNDR', ''
    ]
    
    # Prepare plot data
    plot_list = []
    for m in target_metrics:
        
        ms_val = eval_dict[app1].get(m)
        #######################################################
        """ 
        # Do not show Gower's metric since there are two versions, might be confusing
        if m == 'dcr_gower_distance':
            if not m in eval_dict[app2]:
                if 'dcr_gower' in eval_dict[app2]:
                    m = 'dcr_gower'
                    print(f"In no_privacy, Gower's dcr is called {m}")
                else:
                    print("Check the name for Gower's dcr and add option here.")
        """
        ########################################################
        np_val = eval_dict[app2].get(m)
        if no_categorical and m == 'js_similarity_categorical':
            ms_val = 0
            np_val = 0
        if m == 'gower_distance':  # Do not show gower to make space for legend
            ms_val = 0
            np_val = 0
        plot_list.append({'Metric': m, app1: ms_val, app2: np_val})

    df_filtered = pd.DataFrame(plot_list).set_index('Metric')

    # Plotting
    ax = df_filtered.plot(kind='bar', figsize=(12, 6), width=0.8, color=["#a2d2ff", "#ffccac"])
    plt.title(f'{app1} vs TabDDPM ({dataset}): {comment}')
    plt.ylabel('Value')
    plt.xticks(ticks=[0, 1, 2, 3, 4, 5, 6], labels=target_labels, fontsize=18, rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend(loc='lower right')
    plt.tight_layout()
    save_dir = os.path.join("../privacy_result", f"plots/ablation_{app1}")
    if not os.path.exists(save_dir):
        os.mkdir(save_dir)
    plt.savefig(os.path.join(save_dir, f"{dataset}_{app1}_{app2}_ablation.png")) # Saves the plot to a file
    plt.show()

app1 = "multi_state"
app2 = "no_privacy"
dataset = 'gesture'
eval_dict = load_eval_files(dataset, [app1, app2])
# !!! Adjust no_categorical parameter if the dataset has no categorical features, set to True!!!
plot_two_approaches(app1, app2, eval_dict, dataset, "Privacy-Similarity Tradeoff", no_categorical=True)

## The Basic Distribution Plot (1D)

In [ ]:

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# Organize data into a dictionary for easy looping

plt.rcParams.update({"font.size": 18})

def distribution_map(dataset):
    
    NO_CAT = ["california", "gesture", "wilt", "winequality"]
   
    
    for t in ["num", "cat"]: 
    
        if t == "cat" and dataset in NO_CAT:
            continue
        
        datafile_name=f'X_{t}_train.npy'
        
        # Load your data
        real_data = np.load(f'../data/{dataset}/{datafile_name}', allow_pickle=True)
        syn_data = np.load(f'../privacy_result/{dataset}/ddpm_cb_best_dynamic_adaptive_both/{datafile_name}', allow_pickle=True)
        syn_data_CTABGANPlus = np.load(f'../exp/{dataset}/ctabgan-plus/{datafile_name}', allow_pickle=True)
        syn_data_SMOTE = np.load(f'../exp/{dataset}/smote/{datafile_name}', allow_pickle=True)
        syn_data_TVAE = np.load(f'../exp/{dataset}/tvae/{datafile_name}', allow_pickle=True)
        syn_data_CTGAN = np.load(f'../exp/{dataset}/ctgan/{datafile_name}', allow_pickle=True)
        syn_data_tabddpm = np.load(f'../privacy_result/{dataset}/ddpm_cb_best_no_privacy/{datafile_name}', allow_pickle=True)

        limit = 5
        if real_data.shape[1] > limit:
            real_data = real_data[:, :limit]
            syn_data = syn_data[:, :limit]
            syn_data_CTABGANPlus = syn_data_CTABGANPlus[:, :limit]
            syn_data_SMOTE = syn_data_SMOTE[:, :limit]
            syn_data_TVAE = syn_data_TVAE[:, :limit]
            syn_data_CTGAN = syn_data_CTGAN[:, :limit]
            syn_data_tabddpm = syn_data_tabddpm[:, :limit]


        num_features = real_data.shape[1]
        
        data_dict = {
            "Real": real_data,
            "SMOTE": syn_data_SMOTE,
            "TVAE": syn_data_TVAE,
            "CTGAN": syn_data_CTGAN,
            "CTAB-GAN+": syn_data_CTABGANPlus,
            "Tab-DDPM": syn_data_tabddpm,
            "SynTabRL": syn_data
        }

        approaches = list(data_dict.keys())
        # num_approaches = len(approaches) # If we included "Real" in the visualization
        plot_approaches = [a for a in approaches if a != "Real"] # Exclude "Real" from the list for plotting
        num_approaches = len(plot_approaches) # Exclude "Real" from the count for row labeling
    
    

        # Create the grid (Rows = Approaches, Cols = Features)
        # We adjust figsize so it's wide enough for features and tall enough for rows
        fig, axes = plt.subplots(nrows=num_approaches, 
                                ncols=num_features, 
                                figsize=(num_features * 4, num_approaches * 3),
                                sharex='col',
                                squeeze=False) # Share X axis per column for easier comparison
        
        # for r, approach_name in enumerate(approaches):  # Loop through all approaches including "Real", show Real data distribution at top row
        for r, approach_name in enumerate(plot_approaches):
            current_syn_data = data_dict[approach_name]
            
            
            for c in range(num_features):
                ax = axes[r, c]
                column_data = real_data[:, c]
                num_unique = len(np.unique(column_data))
                
                # Use histplot with 'kde=True' for a hybrid approach
                # Or 'discrete=True' if it's strictly categorical
                
                if approach_name != "Real":
                    
                    sns.histplot(real_data[:, c], ax=ax, fill=True, color='#7FB3D5', 
                            label='Real', stat="density", alpha=0.3, element="step")
                
                sns.histplot(current_syn_data[:, c], ax=ax, color="#57A3D6" if approach_name == "Real" else "#F28888", 
                            label='Synthetic', stat="density", fill=True, element="step")
            
                        
                # --- Formatting ---
                # Dont show x-labels if too many categorical features
                if t == "cat" and num_unique > 10 and isinstance(real_data[0, c], str):
                    ax.set_xticklabels([])   # Removes the label
                if t == "cat" and num_unique > 15:
                    ax.set_xticklabels([])   # Removes the label
                # Only set titles on the top row to identify features
                if r == 0:
                    ax.set_title(f"Feature {c}", fontsize=20, fontweight='bold')
                    
                # Only set y-labels on the first column to identify approaches
                if c == 0:
                    ax.set_ylabel(approach_name, fontsize=20, fontweight='bold')
                else:
                    ax.set_ylabel("") # Remove "Density" label from inner plots
                    
                # Remove individual legends to keep it clean, or add one small one
                ax.get_legend().remove() if ax.get_legend() else None

        # Final adjustments
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.3, wspace=0.2) # Add spacing between plots
        fig.autofmt_xdate(rotation=45)
        fig.suptitle(f"Distribution Map: {dataset} ({t})", 
             fontsize=24, 
             fontweight='bold', 
             y=1.02) # 'y' pushes it slightly above the top row
        plot_path = f"../privacy_result/plots/distribution_maps/{dataset}_{t}_distribution_map.png"
        plt.savefig(plot_path, dpi=300)
        plt.show()
    



datasets = ["abalone", "buddy", "california", "churn2", "gesture", "insurance", "wilt", "bike", "loan", "winequality"]            
for dataset in datasets:
    distribution_map(dataset)


## Correlation Heatmap

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({"font.size": 18})

# Load your data (Replace with your actual file paths)
NO_CAT = ["california", "gesture", "wilt", "winequality"]

def correlation_heatmap(dataset):

    for t in ["num", "cat"]:
        
        if t == "cat" and dataset in NO_CAT:
            continue
        datafile_name=f'X_{t}_train.npy'
        
        # Load your data
        real_data = np.load(f'../data/{dataset}/{datafile_name}', allow_pickle=True)
        syn_data = np.load(f'../privacy_result/{dataset}/ddpm_cb_best_dynamic_adaptive_both/{datafile_name}', allow_pickle=True)
        syn_data_CTABGANPlus = np.load(f'../exp/{dataset}/ctabgan-plus/{datafile_name}', allow_pickle=True)
        syn_data_SMOTE = np.load(f'../exp/{dataset}/smote/{datafile_name}', allow_pickle=True)
        syn_data_TVAE = np.load(f'../exp/{dataset}/tvae/{datafile_name}', allow_pickle=True)
        syn_data_tabddpm = np.load(f'../privacy_result/{dataset}/ddpm_cb_best_no_privacy/{datafile_name}', allow_pickle=True)
        
        limit = 8
        if real_data.shape[1] > limit:
            real_data = real_data[:, :limit]
            syn_data = syn_data[:, :limit]
            syn_data_CTABGANPlus = syn_data_CTABGANPlus[:, :limit]
            syn_data_SMOTE = syn_data_SMOTE[:, :limit]
            syn_data_TVAE = syn_data_TVAE[:, :limit]
            syn_data_tabddpm = syn_data_tabddpm[:, :limit]


        num_data_dict = {
            "Real": real_data,
            "SMOTE": syn_data_SMOTE,
            "TVAE": syn_data_TVAE,
            "CTAB-GAN+": syn_data_CTABGANPlus,
            "Tab-DDPM": syn_data_tabddpm,
            "SynTabRL": syn_data
        }
        
        
        fig, axes = plt.subplots(2, 3, figsize=(26, 16)) 
        axes = axes.flatten()
        
        
        # Setup Ground Truth (Real Data)
        df_real = pd.DataFrame(num_data_dict["Real"])
        if t == "cat": # Encode if dealing with categorical strings
            df_real = df_real.apply(lambda x: pd.factorize(x)[0])
        corr_real = df_real.corr().values

        # Setup Figure (5 synthetic datasets to compare)
        fig, axes = plt.subplots(2, 3, figsize=(32, 20))
        axes = axes.flatten()

        # Filter out "Real" from your dictionary for the comparison loop
        synth_models = {k: v for k, v in num_data_dict.items() if k != "Real"}

        for i, (name, data) in enumerate(synth_models.items()):
            # Calculate Synthetic Correlation
            df_syn = pd.DataFrame(data)
            if t == "cat":
                df_syn = df_syn.apply(lambda x: pd.factorize(x)[0])
            corr_syn = df_syn.corr().values
            
            # COMPUTE ABSOLUTE DIFFERENCE
            diff_matrix = np.abs(corr_real - corr_syn)
            
            # Optional: Zero out the diagonal so it doesn't distract (it will always be 1-1=0)
            np.fill_diagonal(diff_matrix, 0)
            
            # Plot
            sns.heatmap(diff_matrix, 
                        annot=True, 
                        fmt=".2g", 
                        cmap='BuGn', # 'Reds' is better for "error" or "difference" maps
                        ax=axes[i], 
                        vmin=0, vmax=0.5, # Set a lower vmax to highlight smaller errors
                        linewidths=1.5)
            
            axes[i].set_title(f'Abs Diff: Real vs {name}', fontsize=20)

        # Remove the 6th empty subplot if you only have 5 synthetic models
        if len(synth_models) < len(axes):
            fig.delaxes(axes[-1])

        fig.suptitle(f"Correlation Heatmap: {dataset} ({t})", 
             fontsize=24, 
             fontweight='bold', 
             y=1.02) # 'y' pushes it slightly above the top row
        plt.tight_layout()
        plt.savefig(
            f'../privacy_result/plots/correlation_heatmaps/Correlation_Diff_{dataset}_{t}.png', 
            dpi=300,                # High resolution
            bbox_inches='tight',    # Prevents cutting off labels
            facecolor='white',      # Ensures background isn't transparent
            edgecolor='none'
        )
        
        plt.show()



datasets = ["abalone", "buddy", "california", "churn2", "gesture", "insurance", "wilt", "bike", "loan", "winequality"]            
for d in datasets:
    correlation_heatmap(d)        
        


## Data Similarity Visualization

In [ ]:

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.rcParams.update({'font.size': 18})

# Data Preparation
data = {
    'Abalone': [0.011428, 0.030617, 0.023311, 0.001865, 0.025383, 0.007545, 0.013146],
    'Buddy': [0.002658, 0.015356, 0.007298, 0.004695, 0.033896, 0.013138, 0.029475],
    'California': [0.001829, 0.005932, 0.01641, 0.001939, 0.008663, 0.005493, 0.009226],
    'Churn2': [0.004332, 0.079673, 0.050983, 0.006505, 0.03461, 0.027089, 0.033937],
    'Gesture': [0.010285, 0.010722, 0.013567, 0.004153, 0.006195, 0.007346, 0.009541],
    'Insurance': [0.008076, 0.072948, 0.012887, 0.001934, 0.012301, 0.073314, 0.02797],
    'Wilt': [0.002457, 0.067574, 0.00511, 0.001935, 0.012301, 0.010975, 0.009336],
    'Bike': [0.003627, 0.017239, 0.00633, 0.004873, 0.019295, 0.017675, 0.019498],
    'Loan': [0.004866, 0.051455, 0.021017, 0.00339, 0.018112, 0.007644, 0.01042],
    'WQ': [0.014863, 0.087753, 0.132512, 0.005315, 0.037199, 0.00819, 0.018513]
}

approaches = ['TabDDPM', 'SynTabRL sum', 'SynTabRL DCR', 'SMOTE', 'TVAE', 'CTABGAN+', 'CTGAN']
df = pd.DataFrame(data, index=approaches)

# Plotting
plt.figure(figsize=(12, 7))

# Using a reversed color map (YlGnBu_r) so that smaller values look "better" (lighter/greener)
ax = sns.heatmap(df, annot=False, cmap="YlGnBu", fmt=".4f", cbar_kws={'label': 'Wasserstein Distance'})

# Add "Lower is better" indicator
"""
plt.annotate('Lower is better', xy=(1.08, 0.1), xycoords='axes fraction', 
             xytext=(1.08, 0.9), textcoords='axes fraction',
             arrowprops=dict(arrowstyle='<-', color='black', lw=2),
             ha='center', va='center', rotation=90, fontweight='bold')
"""
plt.title('Numerical Similarity: Wasserstein Distance ($\downarrow$)', fontsize=24, pad=20)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Data Preparation from Table 2
data = {
    'AB': [0.995867, 0.991736, 0.989885, 0.997449, 0.995174, 0.988010, 0.998543],
    'BU': [0.994124, 0.992884, 0.994364, 0.982602, 0.978578, 0.993442, 0.970583],
    'CH': [0.988552, 0.979397, 0.986867, 0.980393, 0.992757, 0.985273, 0.941521],
    'IN': [0.997901, 0.964399, 0.996895, 0.999999, 0.956644, 0.984856, 0.960911],
    'BS': [0.996300, 0.992392, 0.994700, 0.992499, 0.961528, 0.980996, 0.976980],
    'LA': [0.996494, 0.986793, 0.989156, 0.980939, 0.941979, 0.994457, 0.969948]
}

approaches = ['TabDDPM', 'SynTabRL sum', 'SynTabRL DCR', 'SMOTE', 'TVAE', 'CTABGAN+', 'CTGAN']
df = pd.DataFrame(data, index=approaches)

# Plotting
plt.figure(figsize=(12, 7))

# Using the YlGnBu colormap (Yellow-Green-Blue)
# Since all values are very high (>0.94), we can adjust vmin to see more contrast
ax = sns.heatmap(df, annot=False, cmap="YlGnBu", fmt=".4f", 
                 cbar_kws={'label': 'JS-Similarity Score'},
                 vmin=0.94, vmax=1.0)


plt.title(r'Categorical Similarity: JS-Similarity ($\uparrow$)', fontsize=24, pad=20)

plt.tight_layout()
plt.show()

## Data Privacy Comparison

Visualizes the DCR of synthetic data generated by various frameworks for performance comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plt.rcParams.update({'font.size': 18})

dcr_data = {
    'TabDDPM': [0.067342, 0.085782, 0.046098, 0.163314, 0.077756, 0.053235, 0.022778, 0.071475, 0.137571, 0.172799],
    'SynTabRL sum': [0.092237, 0.092478, 0.062738, 0.250356, 0.114359, 0.087879, 0.086981, 0.092018, 0.217831, 0.380713],
    'SynTabRL DCR': [0.087573, 0.088925, 0.098112, 0.229888, 0.156131, 0.061423, 0.025814, 0.085472, 0.169793, 0.515812],
    'SMOTE': [0.01831, 0.039905, 0.016331, 0.095773, 0.040984, 0.049092, 0.009378, 0.059419, 0.065679, 0.047248],
    'TVAE': [0.096407, 0.08487, 0.048781, 0.220523, 0.168657, 0.021375, 0.021375, 0.139278, 0.109327, 0.140765],
    'CTABGAN+': [0.08108, 0.092872, 0.070712, 0.214892, 0.162338, 0.128966, 0.024119, 0.171511, 0.132627, 0.168962],
    'CTGAN': [0.115489, 0.111536, 0.061904, 0.27285, 0.204404, 0.16159, 0.029708, 0.188105, 0.147868, 0.19693]
}

nndr_data = {
    'TabDDPM': [0.851049, 0.805354, 0.838897, 0.718445, 0.892021, 0.504559, 0.800493, 0.70544, 0.860056, 0.904137],
    'SynTabRL sum': [0.841386, 0.818774, 0.850776, 0.77106, 0.896981, 0.506677, 0.788272, 0.75756, 0.864634, 0.910538],
    'SynTabRL DCR': [0.847437, 0.814406, 0.858533, 0.777287, 0.903638, 0.547053, 0.803064, 0.760439, 0.858292, 0.914944],
    'SMOTE': [0.432704, 0.521665, 0.442949, 0.483946, 0.480419, 0.51398, 0.51398, 0.653469, 0.541939, 0.58609],
    'TVAE': [0.912819, 0.822397, 0.866599, 0.838947, 0.954742, 0.812505, 0.812505, 0.896698, 0.848544, 0.907666],
    'CTABGAN+': [0.899713, 0.819576, 0.879355, 0.825011, 0.946345, 0.72062, 0.813181, 0.91624, 0.866317, 0.913446],
    'CTGAN': [0.916787, 0.840008, 0.0881, 0.860965, 0.961583, 0.78358, 0.81884, 0.844481, 0.873673, 0.91597]
}

gower_data = {
    'TabDDPM': [0.001986, 0.002328, 0.00158, 0.001246, 0.016329, 0.008418, 0.006079, 0.07935, 0.041109, 0.027814],
    'SynTabRL sum': [0.014477, 0.061797, 0.01471, 0.052161, 0.016657, 0.021762, 0.00653, 0.081763, 0.042855, 0.029026],
    'SynTabRL DCR': [0.014735, 0.060724, 0.014946, 0.050157, 0.016742, 0.017462, 0.006247, 0.080604, 0.041439, 0.029711],
    'SMOTE': [0.008937, 0.062814, 0.012328, 0.041667, 0.013222, 0.004484, 0.004484, 0.081348, 0.040494, 0.019147],
    'TVAE': [0.022819, 0.069772, 0.013905, 0.049046, 0.019414, 0.007455, 0.007455, 0.103711, 0.050965, 0.030856],
    'CTABGAN+': [0.017272, 0.063223, 0.019158, 0.048849, 0.019378, 0.027953, 0.006937, 0.10779, 0.041776, 0.033561],
    'CTGAN': [0.021063, 0.07192, 0.017845, 0.055117, 0.022165, 0.032009, 0.00793, 0.010915, 0.045762, 0.035669]
}



def plot_bars(ax, approach, data_dict, title, ylabel, ylim=None):
    ax.bar(x - width, data_dict[approach], width, label=approach, color=colors[0], edgecolor='grey', linewidth=0.5)
    ax.bar(x, data_dict['SynTabRL sum'], width, label='SynTabRL sum', color=colors[1], edgecolor='grey', linewidth=0.5)
    ax.bar(x + width, data_dict['SynTabRL DCR'], width, label='SynTabRL DCR', color=colors[2], edgecolor='grey', linewidth=0.5)
    
    ax.set_title(title, fontsize=24, pad=15, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=20)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha='right', fontsize=18)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    if ylim: ax.set_ylim(ylim)
    ax.legend(fontsize=18, loc='upper left')



# Visualization settings
colors = ['#FFDAB9', '#D3D3D3', '#ADD8E6']
datasets = ["Abalone", "Buddy", "California", "Churn2", "Gesture", "Insurance", "Wilt", "Bike", "Loan", "Wine Quality"]
x = np.arange(len(datasets))
width = 0.25

def populate_axes(ax, approach, data_dict, title, ylabel, ylim=None):
    # Plotting the three bars for the specific subplot
    ax.bar(x - width, data_dict[approach], width, label=approach, color=colors[0], edgecolor='grey', linewidth=0.5)
    ax.bar(x, data_dict['SynTabRL sum'], width, label='SynTabRL sum', color=colors[1], edgecolor='grey', linewidth=0.5)
    ax.bar(x + width, data_dict['SynTabRL DCR'], width, label='SynTabRL DCR', color=colors[2], edgecolor='grey', linewidth=0.5)
    
    # Formatting
    ax.set_title(title, fontsize=20, pad=15, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=16)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=45, ha='right', fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    if ylim: ax.set_ylim(ylim)
    ax.legend(fontsize=12)
    
    
    
    

# Loop through each baseline approach
for approach in ['TabDDPM', 'SMOTE', 'TVAE', 'CTABGAN+', 'CTGAN']:
    # Create a new figure for each baseline comparison
    
    
    
    
    # Single metric visualization
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))  # Just one metric per figure
    plot_bars(ax, approach, dcr_data, 'DCR Comparison', 'DCR Score')
    
    
    
    # Visualize all three metrics side by side
    # fig, axes = plt.subplots(1, 3, figsize=(24, 8))  # Visualize all three metrics side by side
    # populate_axes(axes[0], approach, dcr_data, 'DCR Comparison', 'DCR Score')
    # populate_axes(axes[1], approach, nndr_data, 'NNDR Comparison', 'NNDR Score', ylim=(0.0, 1.1))
    # populate_axes(axes[2], approach, gower_data, "Gower's DCR Comparison", 'Gower Distance')
    
    fig.suptitle(f'Privacy Metric Comparison: {approach} vs SynTabRL Variants', fontsize=26, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.show()